# User Experience para ciência de dados
Previsão de atrasos em pedidos.

In [ ]:
import pandas as pd

# from ydata_profiling import ProfileReport

df = pd.read_csv('data/amostra.csv')

## Limpeza e tratamento de dados

In [8]:
import re
import unicodedata

# Convert date columns used for feature engineering
for col in ['dt_pagamento_pedido', 'dt_despacho_pedido', 'dt_previsao_entrega_cliente']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# New features
df['dias_gastos_cd'] = (df['dt_despacho_pedido'] - df['dt_pagamento_pedido']).dt.days
df['dias_restantes_prazo'] = (df['dt_previsao_entrega_cliente'] - df['dt_pagamento_pedido']).dt.days

# Normalize city names: case, accents, punctuation and extra spaces
if 'cidade_destinatario' in df.columns:
    def normalize_city(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().lower()
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text.upper()

    df['cidade_destinatario'] = df['cidade_destinatario'].apply(normalize_city).astype('string')

# Binary encode delivery performance
if 'tp_performance_entrega' in df.columns:
    df['tp_performance_entrega'] = (
        df['tp_performance_entrega']
        .astype('string')
        .str.strip()
        .map({
            'Entregue no Prazo': 1,
            'Fora do Prazo': 0
        })
        .astype('Int64')
    )

df = df.drop(
    columns=[
        'row_id',
        'hr_despacho_pedido',
        'dt_entrega_pedido',
        'hr_entrega_pedido',
        'flg_existem_ocorrencias'
    ],
    errors='ignore'
 )

categorical_features = ['uf', 'grp_transportadora', 'cidade_destinatario', 'tp_praca', 'des_unidade_negocio', 'des_cd_origem']

for col in categorical_features:
    df[col] = df[col].astype('category')

df.head(10)

,cod_pedido,cidade_destinatario,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,des_cd_origem,qtd_dias_tat,tp_performance_entrega,dias_gastos_cd,dias_restantes_prazo
0,127995108-1,CURITIBA,PR,Transportadora 4,2023-12-04,2023-12-07,2023-11-30,2023-11-30,Capital,Multi,PR-Campina G. Sul,5.0,1,4.0,7.0
1,126433711-1,PARAUAPEBAS,PA,Transportadora 2,2023-11-21,2023-12-11,2023-11-15,2023-11-17,Interior,Mono,PR-Campina G. Sul,10.0,1,4.0,24.0
2,125177091-1,JACAREZINHO,PR,Transportadora 4,2023-10-28,2023-11-06,2023-10-26,2023-10-27,Interior,Mono,PR-Campina G. Sul,2.0,1,1.0,10.0
3,123168303-1,SAO PAULO,SP,Transportadora 5,2023-08-23,2023-08-24,2023-08-22,2023-08-22,Capital,Multi,PR-Campina G. Sul,2.0,1,1.0,2.0
4,122901808,SAO LUIS,MA,Transportadora 3,2023-08-14,2023-08-24,2023-08-13,2023-08-12,Capital,Multi,PR-Campina G. Sul,8.0,1,2.0,12.0
5,122916223-1,MOGI DAS CRUZES,SP,Transportadora 1,2023-08-14,2023-08-16,2023-08-13,2023-08-13,Reg. Metropolitana,Mono,SP-Registro,3.0,1,1.0,3.0
6,127416977-2,SAO PAULO,SP,Transportadora 1,2023-11-26,2023-12-06,2023-11-24,2023-11-24,Capital,Multi,PR-Campina G. Sul,2.0,1,2.0,12.0
7,125100256-1,CURITIBA,PR,Transportadora 4,2023-10-25,2023-10-26,2023-10-24,2023-10-24,Capital,Mono,PR-Campina G. Sul,2.0,1,1.0,2.0
8,122586320,BRASILIA,DF,Transportadora 4,2023-08-03,2023-08-09,2023-08-02,2023-08-02,Capital,Multi,PR-Campina G. Sul,2.0,1,1.0,7.0
9,123978336-1,BARUERI,SP,Transportadora 3,2023-09-21,2023-09-27,2023-09-19,2023-09-19,Reg. Metropolitana,Multi,PR-Campina G. Sul,3.0,1,2.0,8.0


## Profiling de dados

In [9]:
# profile = ProfileReport(df, title="Profiling Report")
# html = profile.to_html()
# output_file = 'report.html'
# with open(output_file, 'w') as f:
#     f.write(html)

In [10]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
 )

# ---------------------------------------------------------
# Step 3: Split Features (X) and Target (y)
# ---------------------------------------------------------
selected_features = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'dt_previsao_entrega_cliente',
    'dt_criacao',
    'dt_pagamento_pedido',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
    # 'dias_gastos_cd',
    # 'dias_restantes_prazo',
]

available_features = [col for col in selected_features if col in df.columns]
missing_features = [col for col in selected_features if col not in df.columns]

if missing_features:
    print('Missing columns (ignored):', missing_features)

X = df[available_features].copy()
y = df['tp_performance_entrega']

# Remove rows with missing target
valid_mask = y.notna()
X = X.loc[valid_mask].copy()
y = y.loc[valid_mask].astype('int32').copy()

# Convert requested date columns to numeric representation (ordinal days)
date_cols = ['dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido']
for col in [c for c in date_cols if c in X.columns]:
    X[col] = pd.to_datetime(X[col], errors='coerce')
    X[col] = X[col].map(lambda x: x.toordinal() if pd.notna(x) else np.nan).astype('float32')

# Encode requested categorical columns as numeric codes
categorical_cols = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
]
for col in [c for c in categorical_cols if c in X.columns]:
    X[col] = X[col].astype('category').cat.codes.replace(-1, np.nan).astype('float32')

# LightGBM does not accept object/datetime columns directly
unsupported_cols = X.select_dtypes(include=['object', 'datetime64[ns]', 'datetimetz']).columns
if len(unsupported_cols) > 0:
    print('Dropping unsupported columns:', list(unsupported_cols))
X = X.drop(columns=unsupported_cols, errors='ignore')

print('Training columns:', list(X.columns))

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# Step 4: Initialize and Train the Model
# ---------------------------------------------------------
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    is_unbalance=True
)

# Fit model
model.fit(X_train, y_train)

# ---------------------------------------------------------
# Step 5: Make Predictions (The "Risk Score")
# ---------------------------------------------------------
predictions_binary = model.predict(X_test)
predictions_proba = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Step 6: Evaluate Model Quality
# ---------------------------------------------------------
accuracy = accuracy_score(y_test, predictions_binary)
precision = precision_score(y_test, predictions_binary, zero_division=0)
recall = recall_score(y_test, predictions_binary, zero_division=0)
f1 = f1_score(y_test, predictions_binary, zero_division=0)
roc_auc = roc_auc_score(y_test, predictions_proba)
cm = confusion_matrix(y_test, predictions_binary)

print('=== Model Metrics ===')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, predictions_binary, zero_division=0))

Training columns: ['cidade_destinatario', 'uf', 'grp_transportadora', 'dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido', 'tp_praca', 'des_unidade_negocio', 'des_cd_origem']
[LightGBM] [Info] Number of positive: 384620, number of negative: 18205
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002649 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 829
[LightGBM] [Info] Number of data points in the train set: 402825, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.954807 -> initscore=3.050560
[LightGBM] [Info] Start training from score 3.050560
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furth

In [11]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(10, 6))
# lgb.plot_importance(model, ax=ax)
# plt.tight_layout()
# plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
# plt.close(fig)
# print('Saved: feature_importance.png')

In [12]:
# ---------------------------------------------------------
# Step 7: View the Results
# ---------------------------------------------------------
results_df = X_test.copy()
results_df['probabilidade_atraso'] = predictions_proba
results_df['risco_semaforo'] = pd.cut(
    results_df['probabilidade_atraso'],
    bins=[-0.1, 0.3, 0.7, 1.1],
    labels=['🟢 Verde', '🟡 Amarelo', '🔴 Vermelho']
)
print(results_df[['probabilidade_atraso', 'risco_semaforo']].head(20))

        probabilidade_atraso risco_semaforo
406605              0.738127     🔴 Vermelho
445965              0.680910      🟡 Amarelo
456353              0.673734      🟡 Amarelo
88709               0.501959      🟡 Amarelo
355086              0.387354      🟡 Amarelo
391410              0.834255     🔴 Vermelho
41017               0.756021     🔴 Vermelho
256245              0.240326        🟢 Verde
382323              0.670958      🟡 Amarelo
31770               0.481658      🟡 Amarelo
333940              0.751013     🔴 Vermelho
138271              0.592081      🟡 Amarelo
192752              0.607038      🟡 Amarelo
21996               0.622826      🟡 Amarelo
129463              0.620145      🟡 Amarelo
88752               0.541133      🟡 Amarelo
59484               0.505874      🟡 Amarelo
26396               0.460595      🟡 Amarelo
346042              0.551406      🟡 Amarelo
231106              0.780589     🔴 Vermelho


In [13]:
import joblib

# Build categorical mappings from training data
categorical_mappings = {}
for col in [c for c in categorical_cols if c in df.columns]:
    cats = pd.Series(df.loc[valid_mask, col].astype("string").dropna().unique()).sort_values().tolist()
    categorical_mappings[col] = {v: i for i, v in enumerate(cats)}

bundle = {
    "model": model,
    "selected_features": selected_features,
    "date_cols": ["dt_previsao_entrega_cliente", "dt_criacao", "dt_pagamento_pedido"],
    "categorical_cols": [c for c in categorical_cols if c in selected_features],
    "categorical_mappings": categorical_mappings,
    "threshold": 0.5
}

joblib.dump(bundle, "model_bundle.joblib")
print("Saved model_bundle.joblib")

Saved model_bundle.joblib


In [14]:
df[df['tp_performance_entrega'] == 0]

,cod_pedido,cidade_destinatario,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,des_cd_origem,qtd_dias_tat,tp_performance_entrega,dias_gastos_cd,dias_restantes_prazo
480776,123922798-1,SAO PAULO,SP,Transportadora 5,2023-09-20,2023-09-20,2023-09-17,2023-09-17,Capital,Multi,PR-Campina G. Sul,4.0,0,3.0,3.0
480777,127715717-1,ARACAJU,SE,Transportadora 2,2023-11-28,2023-12-07,2023-11-26,2023-11-26,Capital,Mono,PR-Campina G. Sul,12.0,0,2.0,11.0
480778,122505811-1,PORTO ALEGRE,RS,Transportadora 4,2023-07-31,2023-08-02,2023-07-31,2023-07-29,Capital,Multi,PR-Campina G. Sul,4.0,0,2.0,4.0
480779,127406554,BELO HORIZONTE,MG,Transportadora 3,2023-11-27,2023-12-01,2023-11-24,2023-11-24,Capital,Multi,PR-Campina G. Sul,9.0,0,3.0,7.0
480780,125203829,ARAPONGAS,PR,Transportadora 4,2023-10-31,2023-11-01,2023-10-27,2023-10-27,Interior,Mono,PR-Campina G. Sul,4.0,0,4.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503527,124807656-1,NITEROI,RJ,Transportadora 3,2023-10-16,2023-10-18,2023-10-14,2023-10-14,Reg. Metropolitana,Mono,SP-Registro,4.0,0,2.0,4.0
503528,127769981,PARNAMIRIM,RN,Transportadora 1,2023-11-27,2023-12-11,2023-11-26,2023-11-26,Interior,Mono,SP-Registro,NaN,0,1.0,15.0
503529,121957602-1,CANOAS,RS,Transportadora 4,2023-07-11,2023-07-12,2023-07-11,2023-07-09,Capital,Multi,PR-Campina G. Sul,4.0,0,2.0,3.0
503530,123101228-1,GUARULHOS,SP,Transportadora 1,NaT,2023-08-23,2023-08-19,2023-08-19,Reg. Metropolitana,Mono,SP-Registro,4.0,0,NaN,4.0
